In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex

In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

law_title_d = dict(zip(law_df['citation'].tolist(), law_df['title'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

test_df = pd.read_csv("../data/test_rewrite_001.csv")
_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

print("data loaded.")

data loaded.


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)
# dense_index = DenseIndex(model, "../data/processed/_dense_court_removed_citation", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

DenseIndex.embeddings:  (2107648, 1024)


In [4]:
reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)
# reranker = FlagReranker('../ft_data/merged_reranker', use_fp16=True, normalize=True)

In [5]:
valid_df = pd.read_csv("../data/valid_rewrite_001.csv")
valid_id_2_gold_citations_d = {}
valid_id_2_query_d = {}
for query_id, gold_citations, query in zip(valid_df['query_id'], valid_df['gold_citations'], valid_df['query2']):
    valid_id_2_gold_citations_d[query_id] = gold_citations.split(';')
    valid_id_2_query_d[query_id] = query

In [6]:
%load_ext autoreload

%autoreload 2

import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm
import bge_utils
import numpy as np

valid_multi_question_df = pd.read_csv("../data/valid_rewrite_002.csv")

recall_citations_l = []
gold_citations_l = []

query_id_2_query_list = defaultdict(list)

for query_id, query in zip(valid_multi_question_df['query_id'], valid_multi_question_df['query']):
    query_id_2_query_list[query_id].append(query)

for query_id, query in valid_id_2_query_d.items():
    query_id_2_query_list[query_id].append(query) # 完整的问题在最后一个

query_id_2_recall_hits = {}

recall_hits_l = []

cc_repeat_count = defaultdict(int)

for query_id, query_list in query_id_2_query_list.items():
    
    _hits = []
    for q in query_list:
        hits1 = dense_index.search_with_score(q, top_k=100)
        for hit,_ in hits1:
            cc_repeat_count[hit['citation']] += 1
        hits2 = sparse_index.search_with_score(q, top_k=100)
        for hit,_ in hits2:
            cc_repeat_count[hit['citation']] += 1
        _hits = hits_utils.merge_hits_with_score_l_by_max(_hits,hits_utils.merge_hits_with_score_l_by_max(hits1, hits2))
    
    raw_query = query_list[-1]             

    all_hits = _hits.copy()
    
    print(f"[{query_id}] hits_merge.len:", len(all_hits))

    gold_citations_l.append(valid_id_2_gold_citations_d[query_id])

    query_id_2_recall_hits[query_id] = all_hits.copy()
    
    second_layer = citation_utils.compute_citation_score_with_sentence_pos(query_id_2_recall_hits[query_id], decay="log")

    citations = []
    for citation, score in second_layer:
        if citation in court_consideration_d:
            citations.append(citation)
        if citation in law_d:
            citations.append(citation)

    recall_citations_l.append(list(set(citations)))
    
r = metric_utils.cal_recall(recall_citations_l, gold_citations_l)
print(r)

# 准备好document
# recall_hits_l = []
# for recall_citations in recall_citations_l:
#     recall_hits = []
#     for citation in set(recall_citations):
#         if citation in court_consideration_d:
#             recall_hits.append({'citation':citation, 'text':court_consideration_d[citation]})
#         elif citation in law_d:
#             recall_hits.append({'citation':citation, 'text':law_d[citation]})
#     recall_hits_l.append(recall_hits)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


[val_001] hits_merge.len: 479
[val_002] hits_merge.len: 543
[val_003] hits_merge.len: 643
[val_004] hits_merge.len: 390
[val_005] hits_merge.len: 361
[val_006] hits_merge.len: 402
[val_007] hits_merge.len: 503
[val_008] hits_merge.len: 769
[val_009] hits_merge.len: 480
[val_010] hits_merge.len: 457
0.37723928069802104


In [7]:
import numpy as np

def logsumexp(scores):
    scores = np.array(scores)
    m = np.max(scores)
    return m + np.log(np.sum(np.exp(scores - m)))

In [8]:
import citation_utils
def replace_law_id_to_law_title(text):
    text = citation_utils.normalized_sr(text)
    cited_laws = citation_utils.extract_citations_from_text(text)

    for cited_law in cited_laws:
        if cited_law in law_title_d:
            text = text.replace(cited_law, f"[[{cited_law}:{law_title_d[cited_law]}]]")
            
    return text

In [9]:
import pandas as pd
idf_df = pd.read_csv("../data/idf.csv")
idf_d = {citation:idf for citation, idf in zip(idf_df['citation'], idf_df['idf'])}

In [10]:
import math
from collections import defaultdict


def softmax(xs, alpha=5.0):
    """带温度的 softmax"""
    xs_scaled = [alpha * x for x in xs]
    m = max(xs_scaled)
    exps = [math.exp(x - m) for x in xs_scaled]
    s = sum(exps)
    return [e / s for e in exps]


def compute_citation_scores(
    cc_list,
    idf_dict,
    top_k=20,
    alpha=5.0,   # rerank 温度
    beta=0.1     # 位置衰减
):
    """
    返回：
        {citation_id: score}
    """

    # =========================
    # 1️⃣ 选 Top-K CC（降噪关键）
    # =========================
    cc_list = sorted(cc_list, key=lambda x: x["rerank_score"], reverse=True)[:top_k]

    # =========================
    # 2️⃣ 计算 CC 权重（softmax）
    # =========================
    rerank_scores = [cc["rerank_score"] for cc in cc_list]
    cc_weights = softmax(rerank_scores, alpha=alpha)

    # =========================
    # 3️⃣ 统计 TF + sentence index
    # =========================
    # structure:
    # citation_stats[citation_id][cc_id] = {
    #     "tf": int,
    #     "positions": [idx, ...]
    # }
    citation_stats = defaultdict(lambda: defaultdict(lambda: {"tf": 0, "positions": []}))

    for cc in cc_list:
        cc_id = cc["cc_id"]
        for item in cc["citations"]:
            cid = item["citation_id"]
            idx = item["sentence_idx"]

            citation_stats[cid][cc_id]["tf"] += 1
            citation_stats[cid][cc_id]["positions"].append(idx)

    # =========================
    # 4️⃣ 计算最终 score
    # =========================
    citation_scores = defaultdict(float)

    # 建一个 cc_id -> weight 映射
    cc_weight_map = {
        cc["cc_id"]: w for cc, w in zip(cc_list, cc_weights)
    }

    for cid, cc_dict in citation_stats.items():

        idf = idf_dict.get(cid, 1.0)

        total_score = 0.0

        for cc_id, stats in cc_dict.items():
            tf = stats["tf"]
            positions = stats["positions"]

            # ---- TF（log）----
            tf_score = math.log(1 + tf)

            # ---- 位置权重 ----
            pos_weight = sum(math.exp(-beta * p) for p in positions) / len(positions)

            # ---- CC 权重 ----
            w_d = cc_weight_map.get(cc_id, 0.0)

            total_score += w_d * tf_score * idf * pos_weight

        # =========================
        # 5️⃣ 跨 CC 覆盖度增强
        # =========================
        coverage = len(cc_dict)
        coverage_boost = math.log(1 + coverage)

        citation_scores[cid] = total_score * coverage_boost

    return dict(citation_scores)

In [11]:
import numpy as np
import math
from collections import defaultdict

def normalize_and_aggregate_max(cc_results: list[dict], 
                            cap: float = None, 
                            cc_norm_score_threshold: float = None) -> dict[str, float]:
    """
    cc_results: [
        {
            "cc_id": "cc_001",
            "rerank_score": 0.95,
            "citations": [
                {"citation_id": "BGE_123", "first_sentence_idx": 1},
                {"citation_id": "BGE_456", "first_sentence_idx": 10},
            ]
        },
        ...
    ]
    返回: {citation_id: aggregated_score}
    """

    # Step 1: 对rerank scores做min-max归一化
    scores = np.array([cc["rerank_score"] for cc in cc_results])
    score_min, score_max = scores.min(), scores.max()
    
    if score_max - score_min < 1e-9:
        normalized_scores = np.ones(len(scores))
    else:
        normalized_scores = (scores - score_min) / (score_max - score_min)

    # Step 2: 可选 - softmax归一化（更平滑，抑制极端值）
    # temperature = 1.0  # 调小则更集中，调大则更均匀
    # exp_scores = np.exp((scores - scores.max()) / temperature)
    # normalized_scores = exp_scores / exp_scores.sum()

    # Step 3: 计算每个citation的得分并sum pooling
    citation_scores = defaultdict(float)
    citation_counts = defaultdict(int)  # 记录出现次数，用于分析高频citation
    
    for cc, norm_score in zip(cc_results, normalized_scores):
        if cc_norm_score_threshold is not None and norm_score < cc_norm_score_threshold:
            continue
        for citation in cc["citations"]:
            # print ("===>", citation)
            cit_id = citation["citation_id"]
            first_idx = max(citation["first_sentence_idx"], 1)  # 防止除以0
            decay = 1.0 / math.log(1+first_idx)

            contribution = norm_score * decay * idf_d.get(cit_id, 3.0)
            
            if cap is not None:
                contribution = min(norm_score * decay, cap)
                
            citation_counts[cit_id] += 1

            if cit_id not in citation_scores:
                citation_scores[cit_id] = contribution
            elif citation_scores[cit_id] < contribution:
                citation_scores[cit_id] = contribution

    # Step 4: 对高频citation做频率加权（对应你说的第2个信号）
    # 思路：出现在k篇CC中，额外乘以log(1+k)做boost
    for cit_id in citation_scores:
        k = citation_counts[cit_id]
        freq_boost = np.log1p(k)  # log(1+k)，k=1时boost=0.69，k=10时=2.4
        citation_scores[cit_id] *= freq_boost

    # Step 5: 对最终citation分数再做一次归一化，方便后续threshold截断
    if citation_scores:
        vals = np.array(list(citation_scores.values()))
        val_max = vals.max()
        citation_scores = {
            cit_id: score / val_max
            for cit_id, score in citation_scores.items()
        }

    return dict(citation_scores), dict(citation_counts)

In [12]:
def select_citations_d(citation_scores: dict, threshold: float = 0.1) -> list[str]:
    """
    根据归一化后的分数截断，threshold需要在dev set上调
    F1最优threshold通常在0.05~0.2之间
    """
    return [
        cit_id for cit_id, score in citation_scores.items()
        if score >= threshold
    ]

def select_citations_l(citation_scores: dict, threshold: float = 0.1) -> list[str]:
    """
    根据归一化后的分数截断，threshold需要在dev set上调
    F1最优threshold通常在0.05~0.2之间
    """
    return [
        cit_id for cit_id, score in citation_scores
        if score >= threshold
    ]

In [15]:
%load_ext autoreload

%autoreload 2

import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm

from openai import OpenAI
import deepseek_utils
import bge_utils
import evidence_utils
import numpy as np
import math
import os
import os.path
import pickle

query_id_l = []
result_citation_l = []

for idx, (query_id, query_list) in tqdm(enumerate(query_id_2_query_list.items()), total=len(query_id_2_query_list)):

    recall_hits = query_id_2_recall_hits[query_id]

    # for hit,_ in recall_hits:
    #     hit['text'] = replace_law_id_to_law_title(hit['text'])

    print("===>",query_id, ", recall_hits.len:", len(recall_hits))

    raw_query = query_list[-1]

    cc_with_score_l_raw = None

    pkl_path = f"../data/processed/tmp_cc_with_score_l_raw_{idx}.pkl"
    if os.path.exists(pkl_path):
        with open(pkl_path, "rb") as inf:
            cc_with_score_l_raw = pickle.load(inf)
    else:
        cc_with_score_l_raw = reranker_utils.rerank_by_batch_chunked2(reranker, raw_query, [hit for hit,_ in recall_hits])
        with open(pkl_path, "wb") as of:
            pickle.dump(cc_with_score_l_raw, of)

    print("===>",query_id, ", cc_with_score_l_raw.len:", len(cc_with_score_l_raw))
    
    gold_citation_l = gold_citations_l[idx]

    cc_results = []
    for cc, rerank_score in cc_with_score_l_raw:
        d = {}
        d['cc_id'] = cc['citation']
        d['rerank_score'] = rerank_score
        parsed_cc = citation_utils.parse_cc_output_citations_and_sentences_2(cc['text'])
        citations = []
        for citation, sentence_idx in parsed_cc['citations']:
            citations.append({"citation_id":citation, "sentence_idx":sentence_idx})
        d["citations"] = citations

        cc_results.append(d)

    citation_scores = compute_citation_scores(cc_results, idf_d, top_k=200)

    sorted_citation_l = sorted([(citation,score) for citation, score in citation_scores.items()], key=lambda x:x[1], reverse=True)

    sorted_citation_l = [(citation,score) for citation,score in sorted_citation_l if citation in court_consideration_d or citation in law_d]

    result_citation_l.append(sorted_citation_l)

    print("result_citation_l[-1].len:", len(result_citation_l[-1]))
    
    # break
    
max_limit = None
max_r = None
max_p = None
max_f1 = 0
for limit in [5,10,15,20,25,30,35,40,45]:
    result_citation_l2 = []
    
    for r in result_citation_l:
        result_citation_l2.append([citation for citation, _ in r[:limit]])
        
    result = metric_utils.macro_f1(result_citation_l2, gold_citations_l[:len(result_citation_l2)])
    if max_f1 < result['macro_f1']:
        max_r = result['macro_recall']
        max_p = result['macro_precision']
        max_limit = limit
        max_f1 = result['macro_f1']
print(f"[{max_limit}] r:", max_r, ", p:", max_p, "f1:",max_f1)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


  0%|          | 0/10 [00:00<?, ?it/s]

===> val_001 , recall_hits.len: 479
===> val_001 , cc_with_score_l_raw.len: 479
result_citation_l[-1].len: 52
===> val_002 , recall_hits.len: 543
===> val_002 , cc_with_score_l_raw.len: 543
result_citation_l[-1].len: 89
===> val_003 , recall_hits.len: 643
===> val_003 , cc_with_score_l_raw.len: 643
result_citation_l[-1].len: 63
===> val_004 , recall_hits.len: 390
===> val_004 , cc_with_score_l_raw.len: 390
result_citation_l[-1].len: 127
===> val_005 , recall_hits.len: 361
===> val_005 , cc_with_score_l_raw.len: 361
result_citation_l[-1].len: 168
===> val_006 , recall_hits.len: 402
===> val_006 , cc_with_score_l_raw.len: 402
result_citation_l[-1].len: 174
===> val_007 , recall_hits.len: 503
===> val_007 , cc_with_score_l_raw.len: 503
result_citation_l[-1].len: 341
===> val_008 , recall_hits.len: 769
===> val_008 , cc_with_score_l_raw.len: 769
result_citation_l[-1].len: 161
===> val_009 , recall_hits.len: 480
===> val_009 , cc_with_score_l_raw.len: 480
result_citation_l[-1].len: 144
===>